# CDC-5 — Replication slot & WAL growth ⚠️

**Break → Detect → Fix → Prove.** A logical replication slot is a **named cursor into the WAL**.
Postgres retains every WAL segment from the slot's `restart_lsn` forward **until a consumer advances
the slot**. A healthy consumer (Debezium) advances it continuously → WAL recycles, disk stays flat.
If the consumer stops while writes continue, the slot goes **inactive** and the retained WAL grows
without bound — **the database fills its own disk.** A classic, silent Postgres CDC outage.

This is the ⚠️ flagship pathology of the track — it builds on the slot mechanics from
[CDC-1](../postgres_setup/) and the connector lifecycle from [CDC-2](../connector_bringup/).

**Prerequisite:** `make cdc-up`. **Laptop-safe:** a few thousand small rows = a few **MB** of WAL;
bounded loops, never fills disk; manual slot **and** connector dropped at teardown.

> **Honest caveat.** The real incident is a connector down for *hours* while production writes *GBs*
> of WAL. We deliberately **do not** reproduce that (it would risk your disk). We demonstrate the
> **same mechanism at tiny scale** — `retained_bytes` climbing for an inactive slot, the exact metric
> you alert on in prod. MB, not GB; identical failure mode.

In [ ]:
from common import cdc_helpers as cdc
import json, time

TABLE   = "cdc5_orders"
CONN    = "cdc5-orders"
SLOT    = "cdc5_manual_slot"   # hand-made slot (names allow only [a-z0-9_])

print("Connect REST up:", cdc.connect_up(timeout=40))

# Clean slate so the module re-runs deterministically (idempotent).
cdc.teardown(CONN, TABLE)      # delete connector, drop its slot, drop table, delete topic
cdc.drop_slot(SLOT)            # drop our manual slot from any previous run

cdc.seed_orders(TABLE, n=20)
print("seeded public.%s:" % TABLE,
      cdc.pg_exec(f"SELECT count(*) FROM public.{TABLE}", fetch=True)[0][0], "rows")
print("slots at start:", cdc.list_slots())   # should be [] (or unrelated)

## 1. Break — a hand-made, inactive slot

The cleanest deterministic reproduction (as in CDC-1): create a slot **by hand** so nothing is
consuming it. It is `active=False` from birth — its `restart_lsn` will never move on its own.

In [ ]:
cdc.pg_exec(f"SELECT pg_create_logical_replication_slot('{SLOT}', 'pgoutput')", fetch=True)
s = [x for x in cdc.list_slots() if x["slot_name"] == SLOT][0]
print("manual slot created:", s)
print(f"  active={s['active']}  -> nobody is advancing it; retained_bytes will only grow as we write")

## 2. Detect — `retained_bytes` climbs monotonically

`pg_replication_slots` (via `cdc.list_slots()`) is the source of truth. `retained_bytes =
pg_current_wal_lsn() - restart_lsn` is **the WAL this slot is pinning** — the headline metric.

We write rows in **bounded batches** and snapshot the slot after each. `restart_lsn` is frozen
(inactive slot), but `pg_current_wal_lsn()` advances with every write — so the retained gap grows.
This builds the before→after **Prove it** table.

In [ ]:
def retained(slot=SLOT):
    row = [x for x in cdc.list_slots() if x["slot_name"] == slot]
    return row[0]["retained_bytes"] if row else None

def is_active(slot=SLOT):
    row = [x for x in cdc.list_slots() if x["slot_name"] == slot]
    return row[0]["active"] if row else None

BATCHES, PER_BATCH = 5, 400          # 2,000 tiny rows total -> a few MB of WAL, bounded
next_id = 1000
rows_table = [(0, retained(), is_active())]   # (rows_written, retained_bytes, active)

written = 0
for b in range(BATCHES):
    # One bounded multi-row INSERT per batch (small, deterministic, never fills disk).
    values = ",".join(f"({next_id+i},'cust-{i%7}',{round(10+i*0.5,2)},'NEW')" for i in range(PER_BATCH))
    cdc.pg_exec(f"INSERT INTO public.{TABLE}(id,customer,amount,status) VALUES {values}")
    cdc.pg_exec("SELECT pg_switch_wal()", fetch=True)   # force a WAL boundary so retained moves visibly
    next_id += PER_BATCH
    written += PER_BATCH
    rows_table.append((written, retained(), is_active()))

print("INACTIVE SLOT — WAL retained vs rows written")
print(f"{'rows_written':>14} | {'retained_bytes':>16} | active")
print("-"*44)
for r in rows_table:
    print(f"{str(r[0]):>14} | {str(r[1]):>16} | {r[2]}")
print(f"\nMB retained at end: {rows_table[-1][1]/1_048_576:.3f} MB  (tiny & bounded — never fills disk)")

**Diagnose.** The slot is `active=False`, yet `retained_bytes` rises on every measurement.
Postgres can't recycle that WAL because — as far as it knows — a consumer still needs everything from
`restart_lsn` forward. An *abandoned* slot pins WAL **forever** → disk fills → the primary stops
accepting writes. That's the whole outage, here in miniature.

## 3. Fix (A) — reap the orphaned slot

A slot with no owner must be **dropped**. The instant it's gone, Postgres recycles the WAL it was
pinning — `retained_bytes` for that slot disappears.

In [ ]:
before = retained(SLOT)
cdc.drop_slot(SLOT)                       # = pg_drop_replication_slot(SLOT)
after_present = [x for x in cdc.list_slots() if x["slot_name"] == SLOT]
print(f"retained_bytes before drop: {before} ({before/1_048_576:.3f} MB)")
print("slot present after drop    :", bool(after_present), "-> WAL it pinned is now recyclable (~0 retained)")
print("slots now:", cdc.list_slots())

## 4. The realistic version — connector active → paused → resumed

Now the production shape: a **real Debezium connector** holds the slot. We drive it through its
lifecycle via the Kafka Connect REST API (`/pause`, `/resume`) and print the slot state at each step.

A small local helper hits the Connect REST endpoints (you can also call
`cdc._req("PUT", f"/connectors/{name}/pause")` directly).

In [ ]:
import urllib.request

def connect_pause(name):  return urllib.request.urlopen(
    urllib.request.Request(f"{cdc.CONNECT_URL}/connectors/{name}/pause", method="PUT"), timeout=15).status
def connect_resume(name): return urllib.request.urlopen(
    urllib.request.Request(f"{cdc.CONNECT_URL}/connectors/{name}/resume", method="PUT"), timeout=15).status

def conn_slot():
    rows = [x for x in cdc.list_slots() if x["slot_name"] == cdc._safe_slot(CONN)]
    return rows[0] if rows else None

cfg = cdc.debezium_pg_config(CONN, TABLE, snapshot_mode="initial")
print("register ->", cdc.register_connector(CONN, cfg)["status"])      # 201
print("state    ->", cdc.wait_for_connector(CONN, timeout=60))         # RUNNING
time.sleep(3)
print("RUNNING slot:", conn_slot(), "  <- active=True, consumer is advancing it")

### Pause it — the slot stays but goes inactive (the incident)

Pausing is exactly what a crashed/lagging connector looks like to Postgres: the slot persists,
`active` flips to `False`, and any writes from now on are WAL it must retain.

In [ ]:
print("pause ->", connect_pause(CONN))
time.sleep(3)
paused_before = conn_slot()
print("PAUSED slot (before writes):", paused_before)

# write more rows while the connector is paused (bounded)
values = ",".join(f"({5000+i},'cust-{i%7}',{round(10+i*0.5,2)},'NEW')" for i in range(800))
cdc.pg_exec(f"INSERT INTO public.{TABLE}(id,customer,amount,status) VALUES {values}")
cdc.pg_exec("SELECT pg_switch_wal()", fetch=True)
time.sleep(2)
paused_after = conn_slot()
print("PAUSED slot (after  writes):", paused_after)
print(f"retained grew while paused: {paused_before['retained_bytes']} -> {paused_after['retained_bytes']} bytes (active={paused_after['active']})")

### Resume it — the connector catches up and the slot recycles

Resuming reattaches the consumer; Debezium replays the WAL it missed, advances `restart_lsn`, and the
retained WAL **drops back down**. The cure is one REST call.

In [ ]:
print("resume ->", connect_resume(CONN))
print("state  ->", cdc.wait_for_connector(CONN, timeout=60))   # back to RUNNING
# give it a few seconds to consume the backlog and advance the slot
for _ in range(8):
    time.sleep(2)
    now = conn_slot()
    if now and now["active"] and now["retained_bytes"] <= paused_after["retained_bytes"]:
        break
resumed = conn_slot()
print("RESUMED slot:", resumed)
print(f"retained recycled: {paused_after['retained_bytes']} -> {resumed['retained_bytes']} bytes "
      f"(active={resumed['active']}) — a healthy consumer keeps WAL flat")

## 5. Fix (B) — cap the blast radius with `max_slot_wal_keep_size`

Monitoring + reaping handle the *known* cases. The guardrail for the *unknown* runaway is
**`max_slot_wal_keep_size`** (PG 13+): a ceiling on WAL a slot may pin. Past it, Postgres
**invalidates the slot** instead of running out of disk.

Trade-off: an invalidated slot forces its consumer to **re-snapshot**, so size it generously and
alert *before* the cap. This repo's Postgres ships the **default** — see below.

In [ ]:
val = cdc.pg_exec("SHOW max_slot_wal_keep_size", fetch=True)[0][0]
print("max_slot_wal_keep_size =", val)
if val in ("-1", "-1B", "0"):
    print("  -> '-1' means DISABLED (unbounded WAL retention) — the dangerous default.")
    print("     In prod set e.g. max_slot_wal_keep_size='10GB' so a stuck slot can't take PG down.")
else:
    print("  -> a cap is set; a runaway slot will be invalidated rather than fill the disk.")

## 6. Prove & Takeaways — "in real production…"

**Proved, quantitatively:** inactive manual slot → `retained_bytes` climbs monotonically; `drop_slot`
→ WAL freed; connector → flat (running) → climbing (paused) → recycled (resumed).

- **Alert on `pg_replication_slots`** — page on `active = false` for any slot, and on
  `retained_bytes` / `pg_wal_lsn_diff(pg_current_wal_lsn(), restart_lsn)` crossing a threshold. The
  single most important Postgres-CDC alert.
- **One slot per consumer; reap abandoned slots** — a leftover slot from a deleted connector is
  durable state on the *primary*, an outage waiting to happen.
- **Set `max_slot_wal_keep_size`** so a stuck consumer can't take the database down.
- Ties back to **CDC-1** (what a slot *is*) and forward to monitoring / the incident simulator.

## Teardown

In [ ]:
cdc.drop_slot(SLOT)                       # manual slot (already dropped above; idempotent)
cdc.teardown(CONN, TABLE)                 # delete connector, drop its slot, drop table, delete topic
print("manual slot gone:", not any(x["slot_name"] == SLOT for x in cdc.list_slots()))
print("slots now:", cdc.list_slots(), "| make clean clears generated data")